# Pipeline 2 — XLM-RoBERTa-base fine-tuning

This notebook trains **XLM-RoBERTa-base** on **both HAHA 2026 Subtask 1 and Subtask 2**.

The workflow is shared with the other single-encoder notebook: data loading → text building → tokenization → 5-fold fine-tuning → soft-vote ensemble → TSV/ZIP outputs.

In [ ]:
# Kaggle usually already has most packages. Run this cell if imports fail.
# !pip -q install -r ../requirements.txt
# !pip -q install transformers accelerate sentencepiece scikit-learn tqdm

In [ ]:
# ============================================================
# Configuration
# ============================================================
from pathlib import Path

PIPELINE_NAME = "Pipeline 2 — XLM-RoBERTa-base fine-tuning"
PIPELINE_SLUG = "pipeline2_xlmr"
MODEL_NAME = "xlm-roberta-base"
MODEL_DISPLAY = "XLM-RoBERTa-base"

# Default: train both subtasks in the same notebook.
# To save time while debugging, change this to [1] or [2].
TASKS_TO_RUN = [1, 2]

SEED = 42
N_FOLDS = 5
FAST_DEBUG = False   # True = tiny subset, 2 folds, 1 epoch; useful only for checking code flow
DATA_DIR = None      # None = auto-discover dataset directory
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

BASE_TASK_CONFIGS = {
    1: {
        "name": "Subtask 1 — Satirical News Detection",
        "positive_label": "satirical",
        "label2id": {"real": 0, "satirical": 1},
        "id2label": {0: "real", 1: "satirical"},
        "train_files": ["task1_trial.tsv", "task1_dev_gold.tsv"],
        "test_file": "task1_test.tsv",
        "max_length": 128,
        "batch_size": 8,
        "epochs": 5,
        "learning_rate": 2e-5,
        "warmup_ratio": 0.10,
        "weight_decay": 0.01,
        "gradient_checkpointing": True,
    },
    2: {
        "name": "Subtask 2 — LLM Humor Detection",
        "positive_label": "machine",
        "label2id": {"human": 0, "machine": 1},
        "id2label": {0: "human", 1: "machine"},
        "train_files": ["task2_trial.tsv", "task2_dev_gold.tsv"],
        "test_file": "task2_test.tsv",
        "max_length": 128,
        "batch_size": 8,
        "epochs": 6,
        "learning_rate": 2e-5,
        "warmup_ratio": 0.10,
        "weight_decay": 0.01,
        "gradient_checkpointing": True,
    },
}

print(f"Pipeline: {PIPELINE_NAME}")
print(f"Encoder : {MODEL_DISPLAY} ({MODEL_NAME})")
print(f"Output  : {OUTPUT_ROOT.resolve()}")

In [ ]:
# ============================================================
# Imports and reproducibility
# ============================================================
import os
import gc
import re
import zipfile
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = torch.cuda.is_available()
print("Device:", DEVICE)
print("FP16 enabled:", USE_FP16)

In [ ]:
# ============================================================
# Dataset discovery and preprocessing
# ============================================================
REQUIRED_DATA_FILES = {
    "task1_trial.tsv", "task1_dev_gold.tsv", "task1_test.tsv",
    "task2_trial.tsv", "task2_dev_gold.tsv", "task2_test.tsv",
}

def find_dataset_dir(user_dir=None) -> Path:
    if user_dir is not None:
        p = Path(user_dir)
        if p.exists():
            return p
        raise FileNotFoundError(f"DATA_DIR was set but does not exist: {p}")

    candidates = [
        Path("./dataset"),
        Path("../dataset"),
        Path("/kaggle/working/dataset"),
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend([p for p in kaggle_input.rglob("*") if p.is_dir()])

    for p in candidates:
        if p.exists() and REQUIRED_DATA_FILES.issubset({x.name for x in p.glob("*.tsv")}):
            return p

    raise FileNotFoundError(
        "Could not find dataset directory. Set DATA_DIR to the folder containing the task TSV files."
    )

DATA_PATH = find_dataset_dir(DATA_DIR)
print("Dataset directory:", DATA_PATH.resolve())


def clean_text(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    text = str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_text(row, task_id: int) -> str:
    headline = clean_text(row.get("headline", ""))
    if task_id == 1:
        context = clean_text(row.get("context", ""))
        return headline if not context else f"{headline} [SEP] {context}"
    if task_id == 2:
        joke = clean_text(row.get("joke", ""))
        hint = "[LONG]" if len(joke.split()) >= 18 else "[SHORT]"
        return f"{headline} [SEP] {hint} {joke}"
    raise ValueError(f"Unsupported task_id: {task_id}")


def load_task_data(task_id: int):
    cfg = BASE_TASK_CONFIGS[task_id]
    frames = []
    for filename in cfg["train_files"]:
        path = DATA_PATH / filename
        df_part = pd.read_csv(path, sep="\t")
        df_part["source_file"] = filename
        frames.append(df_part)
    train_df = pd.concat(frames, ignore_index=True)
    test_df = pd.read_csv(DATA_PATH / cfg["test_file"], sep="\t")

    if "tag" not in train_df.columns:
        raise ValueError(f"Missing 'tag' column for task {task_id}")
    train_df["label"] = train_df["tag"].map(cfg["label2id"])
    if train_df["label"].isna().any():
        bad = train_df.loc[train_df["label"].isna(), "tag"].unique().tolist()
        raise ValueError(f"Unknown labels in task {task_id}: {bad}")
    train_df["label"] = train_df["label"].astype(int)

    train_df["text"] = train_df.apply(lambda r: build_text(r, task_id), axis=1)
    test_df["text"] = test_df.apply(lambda r: build_text(r, task_id), axis=1)

    if FAST_DEBUG:
        train_df = train_df.groupby("label", group_keys=False).head(24).reset_index(drop=True)
        test_df = test_df.head(16).reset_index(drop=True)
        cfg = dict(cfg)
        cfg["epochs"] = 1
        cfg["batch_size"] = min(8, cfg["batch_size"])

    print(f"Task {task_id}: train={train_df.shape}, test={test_df.shape}")
    print(train_df["tag"].value_counts().to_dict())
    return train_df, test_df, cfg

for task_id in TASKS_TO_RUN:
    _train_df, _test_df, _cfg = load_task_data(task_id)
    display(_train_df[["id", "tag", "text"]].head(3))

In [ ]:
# ============================================================
# Modeling utilities
# ============================================================
class TextClassificationDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length, labels=None):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
        self.labels = None if labels is None else list(labels)

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def prepare_tokenizer(task_id: int):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    desired_special_tokens = ["[SEP]"]
    if task_id == 2:
        desired_special_tokens.extend(["[LONG]", "[SHORT]"])
    vocab = tokenizer.get_vocab()
    tokens_to_add = [tok for tok in desired_special_tokens if tok not in vocab]
    if tokens_to_add:
        tokenizer.add_special_tokens({"additional_special_tokens": tokens_to_add})
    return tokenizer, len(tokens_to_add)


def make_loader(texts, labels, tokenizer, max_length, batch_size, shuffle=False):
    dataset = TextClassificationDataset(texts, tokenizer, max_length, labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=2, pin_memory=torch.cuda.is_available())


def make_test_loader(texts, tokenizer, max_length, batch_size):
    dataset = TextClassificationDataset(texts, tokenizer, max_length, labels=None)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())


def move_batch_to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}


def positive_f1(y_true, prob):
    pred = prob.argmax(axis=1)
    return f1_score(y_true, pred, pos_label=1)


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        outputs = model(**batch)
        probs = torch.softmax(outputs.logits, dim=-1)
        all_probs.append(probs.detach().cpu().numpy())
    return np.concatenate(all_probs, axis=0)


def train_one_fold(task_id, fold, train_part, val_part, cfg, tokenizer, added_tokens):
    fold_dir = OUTPUT_ROOT / f"{PIPELINE_SLUG}_task{task_id}" / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=cfg["id2label"],
        label2id=cfg["label2id"],
    )
    if added_tokens > 0:
        model.resize_token_embeddings(len(tokenizer))
    if cfg.get("gradient_checkpointing", False):
        model.gradient_checkpointing_enable()
        model.config.use_cache = False
    model.to(DEVICE)

    train_loader = make_loader(
        train_part["text"].tolist(), train_part["label"].tolist(), tokenizer,
        cfg["max_length"], cfg["batch_size"], shuffle=True,
    )
    val_loader = make_loader(
        val_part["text"].tolist(), val_part["label"].tolist(), tokenizer,
        cfg["max_length"], cfg["batch_size"], shuffle=False,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
    )
    total_steps = max(1, len(train_loader) * cfg["epochs"])
    warmup_steps = int(total_steps * cfg["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)

    best_f1 = -1.0
    best_epoch = -1
    history = []

    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        losses = []
        progress = tqdm(train_loader, desc=f"Task {task_id} Fold {fold} Epoch {epoch}", leave=False)
        for batch in progress:
            batch = move_batch_to_device(batch)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_FP16):
                outputs = model(**batch)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(float(loss.detach().cpu()))
            progress.set_postfix(loss=np.mean(losses))

        val_probs = predict_proba(model, val_loader)
        val_f1 = positive_f1(val_part["label"].values, val_probs)
        mean_loss = float(np.mean(losses)) if losses else 0.0
        history.append({"epoch": epoch, "train_loss": mean_loss, "val_f1": val_f1})
        print(f"Task {task_id} Fold {fold} | epoch={epoch} | loss={mean_loss:.4f} | val_f1={val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch
            model.save_pretrained(fold_dir)
            tokenizer.save_pretrained(fold_dir)

    # Reload best checkpoint for validation probabilities.
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    best_model = AutoModelForSequenceClassification.from_pretrained(fold_dir).to(DEVICE)
    val_probs = predict_proba(best_model, val_loader)

    del best_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return fold_dir, val_probs, best_f1, best_epoch, history


def ensemble_predict_from_dirs(fold_dirs, texts, max_length, batch_size):
    all_probs = []
    for fold_dir in fold_dirs:
        tokenizer = AutoTokenizer.from_pretrained(fold_dir, use_fast=True)
        model = AutoModelForSequenceClassification.from_pretrained(fold_dir).to(DEVICE)
        loader = make_test_loader(texts, tokenizer, max_length, batch_size)
        probs = predict_proba(model, loader)
        all_probs.append(probs)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return np.mean(all_probs, axis=0)


def run_transformer_task(task_id: int):
    set_seed(SEED)
    train_df, test_df, cfg = load_task_data(task_id)
    tokenizer, added_tokens = prepare_tokenizer(task_id)

    n_splits = 2 if FAST_DEBUG else N_FOLDS
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_probs = np.zeros((len(train_df), 2), dtype=np.float32)
    fold_dirs, fold_scores, fold_histories = [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["label"]), start=1):
        train_part = train_df.iloc[tr_idx].reset_index(drop=True)
        val_part = train_df.iloc[va_idx].reset_index(drop=True)
        fold_dir, val_probs, best_f1, best_epoch, history = train_one_fold(
            task_id, fold, train_part, val_part, cfg, tokenizer, added_tokens
        )
        oof_probs[va_idx] = val_probs
        fold_dirs.append(fold_dir)
        fold_scores.append(best_f1)
        fold_histories.append(history)
        print(f"Best Task {task_id} Fold {fold}: F1={best_f1:.4f} at epoch {best_epoch}")

    oof_pred = oof_probs.argmax(axis=1)
    oof_f1 = f1_score(train_df["label"], oof_pred, pos_label=1)
    print("=" * 70)
    print(f"{PIPELINE_NAME} | Task {task_id} | OOF positive-class F1: {oof_f1:.4f}")
    print("Fold F1:", [round(x, 4) for x in fold_scores])
    print(classification_report(train_df["label"], oof_pred, target_names=[cfg["id2label"][0], cfg["id2label"][1]]))
    print("Confusion matrix:\n", confusion_matrix(train_df["label"], oof_pred))

    test_probs = ensemble_predict_from_dirs(
        fold_dirs,
        test_df["text"].tolist(),
        cfg["max_length"],
        cfg["batch_size"],
    )
    test_pred = test_probs.argmax(axis=1)
    pred_labels = [cfg["id2label"][int(x)] for x in test_pred]
    pred_path = OUTPUT_ROOT / f"{PIPELINE_SLUG}_task{task_id}_predictions.tsv"
    submission_df = pd.DataFrame({"id": test_df["id"], "tag": pred_labels})
    submission_df.to_csv(pred_path, sep="\t", index=False)
    print(f"Saved predictions: {pred_path}")
    display(submission_df.head())

    return {
        "task_id": task_id,
        "cfg": cfg,
        "train_df": train_df,
        "test_df": test_df,
        "fold_dirs": fold_dirs,
        "fold_scores": fold_scores,
        "fold_histories": fold_histories,
        "oof_probs": oof_probs,
        "oof_f1": oof_f1,
        "test_probs": test_probs,
        "prediction_path": pred_path,
    }

In [ ]:
# ============================================================
# Train selected tasks and create a submission ZIP
# ============================================================
RESULTS = {}
for task_id in TASKS_TO_RUN:
    print("\n" + "#" * 80)
    print(f"Running {PIPELINE_NAME} on Task {task_id}")
    print("#" * 80)
    RESULTS[task_id] = run_transformer_task(task_id)

zip_path = OUTPUT_ROOT / f"{PIPELINE_SLUG}_submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for task_id, result in RESULTS.items():
        # CodaBench-style filename inside zip.
        arcname = f"task{task_id}.tsv"
        zf.write(result["prediction_path"], arcname=arcname)
print(f"Created submission ZIP: {zip_path}")